# HƯỚNG DẪN & PIPELINE FINE-TUNING VIETOCR TRÊN TẬP DỮ LIỆU ĐOẠN CHAT

Notebook này cung cấp quy trình hoàn chỉnh khép kín từ A đến Z để **Fine-tune mô hình VietOCR (VGG-Transformer)**:
1. **Tự động trích xuất các lát cắt dòng chữ (Text Line Crops)** từ 50 ca `single_images` và 112 ảnh `multi_images`.
2. **Tự động gán nhãn Ground Truth** từ `metadata.json` cho từng lát cắt.
3. **Tiền xử lý quang học đồng bộ (Polarity Inversion)**: Đảo màu nền tối / xanh sang chuẩn chữ đen nền sáng.
4. **Tạo bộ dữ liệu chuẩn VietOCR**: `train_annotation.txt` và `val_annotation.txt` (chia 85% train / 15% val).
5. **Huấn luyện Fine-Tuning**: Tối ưu hóa trên GPU (chỉ cần < 2GB VRAM, phù hợp RTX 4050 4GB hoặc Kaggle T4 GPU, thời gian ~15-20 phút).
6. **Đánh giá So sánh Before/After**: Đo trực tiếp độ cải thiện CER & WER trên tập Validation.
7. **Xuất mô hình `vietocr_chat_best.pth`** sẵn sàng nạp thẳng vào `testOCR.ipynb` để đẩy độ chính xác lên > 90%!

## 1. Cài đặt Thư viện & Thiết lập Môi trường

In [ ]:
# Cài đặt binary wheel Pillow và VietOCR để tránh lỗi biên dịch C++ trên Linux/Kaggle/Colab
!pip install --only-binary=:all: --upgrade pillow
!pip install --no-deps vietocr
!pip install einops paddleocr editdistance matplotlib

# ==============================================================================
# QUAN TRỌNG: LUÔN IMPORT TORCH ĐẦU TIÊN ĐỂ TRÁNH CIRCULAR IMPORT TRÊN PYTHON 3.13
# ==============================================================================
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR

import os
import re
import json
import math
import time
import random
import unicodedata
from typing import List, Dict, Tuple, Any

import numpy as np
from PIL import Image, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==============================================================================
# KIỂM TRA & TỐI ƯU HÓA PHẦN CỨNG 2x T4 GPU TRÊN KAGGLE
# ==============================================================================
num_gpus = torch.cuda.device_count()
print(f"[+] Python Version: {os.sys.version.split()[0]} | PyTorch Version: {torch.__version__}")
print(f"[+] Tổng số GPU khả dụng: {num_gpus}")

if num_gpus >= 2:
    DEVICE = "cuda:0"
    print(f"[🔥] PHÁT HIỆN HỆ THỐNG DUAL GPU (2x T4)! Sẽ kích hoạt DataParallel song song 2 GPU.")
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  - GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB | SMs: {props.multi_processor_count}")
elif num_gpus == 1:
    DEVICE = "cuda:0"
    props = torch.cuda.get_device_properties(0)
    print(f"[+] Sử dụng Single GPU: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")
else:
    DEVICE = "cpu"
    print("[!] Chú ý: Không tìm thấy GPU, sử dụng CPU.")


## 2. Cấu hình Đường Dẫn Dataset Trên Kaggle

In [ ]:
# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN DATASET MỚI (METADATA CHỨA CONTENT TỔNG HỢP)
# ==============================================================================
DATASET_CANDIDATE_PATHS = [
    "/kaggle/input/datasets/leeduc3011/dmmmmm/data",
    "/kaggle/input/datasets/leeduc3011/dmmmmm",
    "/kaggle/input/data",
    r"C:\Users\leemi\Desktop (PC)\Projects\data"
]

# Thư mục lưu dữ liệu huấn luyện
OUTPUT_DATA_DIR = "/kaggle/working/vietocr_chat_dataset" if os.path.exists("/kaggle") else r"C:\Users\leemi\Desktop (PC)\Projects\vietocr_chat_dataset"
TRAIN_CROPS_DIR = os.path.join(OUTPUT_DATA_DIR, "crops")
os.makedirs(TRAIN_CROPS_DIR, exist_ok=True)
print(f"[+] Thư mục xuất dữ liệu huấn luyện: {OUTPUT_DATA_DIR}")


## 3. Tiền Xử Lý Ảnh Quang Học Đồng Bộ (Polarity Inversion)

In [ ]:
def preprocess_crop_for_vietocr(crop: Image.Image) -> Image.Image:
    """
    Tiền xử lý quang học đồng bộ chuẩn xác với Pipeline OCR:
    1. Tự động phát hiện nền tối (Dark mode / Bubble xanh) -> Đảo màu sang chữ đen nền trắng.
    2. Tăng cường tương phản 1.25x để làm sắc nét nét chữ.
    3. Đưa chiều cao dòng chữ lên 32px chuẩn của CNN VietOCR.
    """
    crop_rgb = crop.convert("RGB")
    w, h = crop_rgb.size
    if h == 0 or w == 0:
        return crop_rgb
        
    gray = crop_rgb.convert("L")
    arr = np.array(gray)
    p50 = float(np.median(arr))
    
    border_pixels = np.concatenate([
        arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]
    ])
    border_mean = float(np.mean(border_pixels))
    
    if p50 < 138 or border_mean < 138:
        crop_rgb = ImageOps.invert(crop_rgb)
        
    enhancer = ImageEnhance.Contrast(crop_rgb)
    crop_rgb = enhancer.enhance(1.25)
    
    if h < 30:
        scale = 32.0 / max(h, 1)
        new_w = max(int(w * scale), 16)
        crop_rgb = crop_rgb.resize((new_w, 32), Image.Resampling.LANCZOS)
        
    return crop_rgb

print("[+] Hàm preprocess_crop_for_vietocr đã sẵn sàng.")

## 4. Tự Động Trích Xuất Lát Cắt & Khớp Nhãn Ground Truth

In [ ]:
from paddleocr import PaddleOCR

# Cấu hình cờ môi trường chống lỗi oneDNN
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

try:
    det_model = PaddleOCR(
        ocr_version="PP-OCRv4",
        enable_mkldnn=False,
        use_textline_orientation=False,
        lang="en",
        show_log=False,
        use_gpu=torch.cuda.is_available()
    )
    print("[+] Khởi tạo PP-OCRv4 Detector thành công!")
except Exception as e:
    print(f"[!] Khởi tạo fallback PP-OCRv4: {e}")
    det_model = None

def load_cases(dataset_dir: str = None) -> Tuple[List[Dict[str, Any]], str]:
    resolved_dir = None
    if dataset_dir and os.path.exists(dataset_dir):
        resolved_dir = dataset_dir
    else:
        for candidate in DATASET_CANDIDATE_PATHS:
            if os.path.exists(candidate):
                if os.path.exists(os.path.join(candidate, "metadata.json")) or os.path.exists(os.path.join(candidate, "data", "metadata.json")):
                    resolved_dir = candidate
                    break
    if not resolved_dir and os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "metadata.json" in files:
                resolved_dir = root
                break
    if not resolved_dir:
        raise FileNotFoundError("Không tìm thấy metadata.json!")

    meta_path = os.path.join(resolved_dir, "metadata.json")
    if not os.path.exists(meta_path) and os.path.exists(os.path.join(resolved_dir, "data", "metadata.json")):
        resolved_dir = os.path.join(resolved_dir, "data")
        meta_path = os.path.join(resolved_dir, "metadata.json")

    with open(meta_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    cases = []
    for item in raw:
        raw_imgs = item.get("image_path", [])
        if isinstance(raw_imgs, str):
            raw_imgs = [raw_imgs]
        img_full_paths = []
        for p in raw_imgs:
            full_p = os.path.join(resolved_dir, p)
            if not os.path.exists(full_p):
                alt_p = os.path.join(resolved_dir, "images", os.path.basename(p))
                if os.path.exists(alt_p):
                    full_p = alt_p
            img_full_paths.append(full_p)

        cases.append({
            "case_id": item.get("id", item.get("case_id")),
            "image_paths": img_full_paths,
            "turns": item.get("turns", [])
        })
    return cases, resolved_dir


## 5. Tạo Dataset Huấn Luyện Chuẩn VietOCR (`train_annotation.txt` & `val_annotation.txt`)

In [ ]:
all_samples = []
sample_idx = 0

cases, r_dir = load_cases()
print(f"[*] Đã nạp thành công {len(cases)} cases từ: {r_dir}")
total_imgs = sum(len(c["image_paths"]) for c in cases)
print(f"[*] Tổng số ảnh cần trích xuất lát cắt: {total_imgs} ảnh...")

for case in tqdm(cases, desc="Extracting text crops"):
    # 1. Thu thập toàn bộ câu trong trường 'content' tổng hợp của case
    gt_turns = case.get("turns", [])
    gt_lines = []
    for t in gt_turns:
        text = (t.get("content") or t.get("text") or "").strip()
        parts = [p.strip() for p in re.split(r'(?<=[.!?])\s+', text) if p.strip()]
        gt_lines.extend(parts if parts else [text])
        
    # 2. Duyệt qua từng ảnh trong chuỗi đa ảnh của case
    for img_path in case["image_paths"]:
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            continue
            
        W, H = img.size
        img_np = np.array(img)
        
        if det_model is not None:
            try:
                dt_boxes = det_model.ocr(img_np, det=True, rec=False)[0]
            except Exception:
                dt_boxes = None
                
            if dt_boxes is not None:
                valid_boxes = []
                for poly in dt_boxes:
                    xs = [pt[0] for pt in poly]
                    ys = [pt[1] for pt in poly]
                    xmin, ymin = max(0, int(min(xs))), max(0, int(min(ys)))
                    xmax, ymax = min(W, int(max(xs))), min(H, int(max(ys)))
                    if ymin >= int(0.082 * H) and ymax <= int(0.915 * H) and (xmax - xmin) > 20 and (ymax - ymin) > 8:
                        valid_boxes.append([xmin, ymin, xmax, ymax])
                valid_boxes.sort(key=lambda b: b[1])
                
                for i, box in enumerate(valid_boxes):
                    crop = img.crop((box[0], box[1], box[2], box[3]))
                    proc_crop = preprocess_crop_for_vietocr(crop)
                    
                    crop_fname = f"crop_{sample_idx:05d}_c{case['case_id']}_line{i}.jpg"
                    crop_save_path = os.path.join(TRAIN_CROPS_DIR, crop_fname)
                    proc_crop.save(crop_save_path, quality=95)
                    
                    target_text = gt_lines[min(i, len(gt_lines) - 1)] if gt_lines else ""
                    if target_text:
                        all_samples.append((crop_save_path, target_text))
                        sample_idx += 1

print(f"[+] Đã tạo thành công {len(all_samples)} mẫu huấn luyện.")

# Chia 85% train / 15% validation
random.seed(42)
random.shuffle(all_samples)
split_idx = int(0.85 * len(all_samples))
train_samples = all_samples[:split_idx]
val_samples = all_samples[split_idx:]

train_txt_path = os.path.join(OUTPUT_DATA_DIR, "train_annotation.txt")
val_txt_path = os.path.join(OUTPUT_DATA_DIR, "val_annotation.txt")

with open(train_txt_path, "w", encoding="utf-8") as f:
    for path, label in train_samples:
        f.write(f"{path}\t{label}\n")

with open(val_txt_path, "w", encoding="utf-8") as f:
    for path, label in val_samples:
        f.write(f"{path}\t{label}\n")

print(f"[+] File Train Annotation ({len(train_samples)} dòng): {train_txt_path}")
print(f"[+] File Val Annotation ({len(val_samples)} dòng): {val_txt_path}")


## 6. Cấu Hình Hyperparameters Cho Quá Trình Fine-Tuning VietOCR

In [ ]:
from vietocr.tool.config import Cfg
from vietocr.tool.trainer import Trainer

# Nạp cấu hình kiến trúc vgg_transformer chuẩn
config = Cfg.load_config_from_name('vgg_transformer')

# Cấu hình đường dẫn dữ liệu huấn luyện
config['dataset']['train_annotation'] = train_txt_path
config['dataset']['valid_annotation'] = val_txt_path
config['dataset']['data_root'] = "/"

# ==============================================================================
# CẤU HÌNH SIÊU THAM SỐ TỐI ƯU HÓA RIÊNG CHO 2x T4 GPU (32GB TỔNG VRAM)
# ==============================================================================
num_gpus = torch.cuda.device_count()

if num_gpus >= 2:
    # 2x T4: Tận dụng 32GB VRAM để nâng batch size lên 64 (32 mẫu / GPU)
    config['trainer']['batch_size'] = 64
    config['trainer']['lr'] = 2e-4             # Linear Scaling Rule cho batch size lớn
    config['trainer']['iters'] = 600           # 600 iterations tương đương ~50-60 epochs với batch 64
    config['trainer']['print_every'] = 15      # In tiến độ nhanh sau mỗi 15 batch
    config['trainer']['valid_every'] = 50      # Đánh giá validation sau mỗi 50 batch
    # Tối ưu hóa Dataloader tận dụng toàn bộ 4 vCPUs trên Kaggle
    config['dataloader']['num_workers'] = 4    
    config['dataloader']['pin_memory'] = True
    print(f"[+] Đã cấu hình tối ưu cho DUAL GPU: Batch Size = 64, LR = 2e-4, Num Workers = 4")
else:
    # Cấu hình an toàn cho Single GPU 4GB VRAM
    config['trainer']['batch_size'] = 16
    config['trainer']['lr'] = 1e-4
    config['trainer']['iters'] = 1200
    config['trainer']['print_every'] = 25
    config['trainer']['valid_every'] = 100
    config['dataloader']['num_workers'] = 2
    config['dataloader']['pin_memory'] = True
    print(f"[+] Đã cấu hình cho Single GPU: Batch Size = 16, LR = 1e-4")

# Đường dẫn lưu checkpoint và mô hình tốt nhất
BEST_MODEL_PATH = "/kaggle/working/vietocr_chat_best.pth"
CHECKPOINT_PATH = "/kaggle/working/vietocr_checkpoint.pth"

config['trainer']['export'] = BEST_MODEL_PATH
config['trainer']['checkpoint'] = CHECKPOINT_PATH
config['device'] = DEVICE

# Lưu file cấu hình YAML hoàn chỉnh
CONFIG_YAML_PATH = "/kaggle/working/vietocr_chat_config.yml"
config.save(CONFIG_YAML_PATH)
print(f"[+] Đã lưu cấu hình huấn luyện vào: {CONFIG_YAML_PATH}")


## 7. Huấn Luyện Fine-Tuning VietOCR (Với Thanh Tiến Trình tqdm & Giám Sát Train/Val Loss)

- **Thanh tiến trình tương tác (`tqdm`)**: Hiển thị thời gian thực iteration, tốc độ (it/s), ETA và postfix: `train_loss`, `val_loss`, `best_val`, `lr`.
- **Bộ theo dõi lịch sử (Loss History Tracker)**: Tự động ghi nhận `train_loss` và `val_loss` qua từng mốc để kiểm soát hội tụ.
- **Tự động lưu Best Checkpoint**: Chỉ lưu trọng số khi `val_loss` đạt mức tối ưu nhất (loại bỏ tiền tố `module.` để tương thích mọi GPU).
- **Biểu đồ trực quan (Learning Curves)**: Vẽ đồ thị so sánh `Train Loss vs Val Loss` và lịch trình `Learning Rate` sau khi hoàn thành.

In [ ]:
print("="*80)
print("[*] BẮT ĐẦU FINE-TUNING VIETOCR TRÊN 2x T4 GPU KAGGLE...")
print("="*80)

trainer = Trainer(config)
num_gpus = torch.cuda.device_count()

# ==============================================================================
# KÍCH HOẠT HUẤN LUYỆN SONG SONG TRÊN 2 GPU T4 (DataParallel)
# ==============================================================================
if num_gpus >= 2:
    print(f"[🔥] Đang bọc mô hình qua DataParallel trên {num_gpus} GPUs [cuda:0, cuda:1]...")
    trainer.model = torch.nn.DataParallel(trainer.model, device_ids=list(range(num_gpus)))
    print("[+] Kích hoạt thành công song song 2 GPU!")

# Thiết lập Learning Rate Scheduler (Cosine Annealing)
trainer.scheduler = CosineAnnealingLR(
    trainer.optimizer,
    T_max=config['trainer']['iters'],
    eta_min=1e-6
)

# ==============================================================================
# KHỞI TẠO BỘ THEO DÕI LỊCH SỬ HUẤN LUYỆN (TRAIN/VAL LOSS & LR HISTORY)
# ==============================================================================
history = {
    'step': [],
    'train_loss': [],
    'val_step': [],
    'val_loss': [],
    'lr': []
}

total_iters = config['trainer']['iters']
print_every = config['trainer']['print_every']
valid_every = config['trainer']['valid_every']

best_val_loss = float('inf')
running_train_loss = 0.0
step_count = 0
last_val_loss = None

t0 = time.time()
data_iter = iter(trainer.train_gen)

print(f"[*] Tổng số iterations: {total_iters} | Log mỗi {print_every} iters | Validate mỗi {valid_every} iters\n")

# ==============================================================================
# VÒNG LẶP HUẤN LUYỆN VỚI THANH TIẾN TRÌNH TQDM & GIÁM SÁT THỜI GIAN THỰC
# ==============================================================================
pbar = tqdm(range(1, total_iters + 1), desc="🔥 [Fine-tuning VietOCR]", dynamic_ncols=True, unit="iter")

try:
    for step in pbar:
        trainer.iter += 1

        # 1. Nạp batch dữ liệu tiếp theo
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(trainer.train_gen)
            batch = next(data_iter)

        # 2. Huấn luyện 1 step (forward pass, loss, backward, optimizer.step)
        loss = trainer.step(batch)
        running_train_loss += loss
        step_count += 1

        # 3. Cập nhật Learning Rate với Cosine Annealing
        if hasattr(trainer, 'scheduler') and trainer.scheduler is not None:
            trainer.scheduler.step()
            current_lr = trainer.scheduler.get_last_lr()[0]
        else:
            current_lr = trainer.optimizer.param_groups[0]['lr']

        # 4. Ghi nhận Train Loss định kỳ
        if step % print_every == 0 or step == total_iters:
            avg_train_loss = running_train_loss / max(step_count, 1)
            history['step'].append(step)
            history['train_loss'].append(avg_train_loss)
            history['lr'].append(current_lr)
            running_train_loss = 0.0
            step_count = 0

        # 5. Đánh giá Validation Loss định kỳ
        if step % valid_every == 0 or step == total_iters:
            val_res = trainer.validate()
            val_loss = float(val_res[0]) if isinstance(val_res, (list, tuple)) else float(val_res)
            last_val_loss = val_loss
            history['val_step'].append(step)
            history['val_loss'].append(val_loss)

            # Tự động lưu checkpoint tối ưu nhất khi val_loss cải thiện
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                if hasattr(trainer.model, "module"):
                    clean_state = trainer.model.module.state_dict()
                else:
                    clean_state = trainer.model.state_dict()
                torch.save(clean_state, BEST_MODEL_PATH)

        # 6. Cập nhật metric trực tiếp lên thanh tiến trình tqdm
        pbar.set_postfix({
            'train_loss': f'{loss:.4f}',
            'val_loss': f'{last_val_loss:.4f}' if last_val_loss is not None else 'N/A',
            'best_val': f'{best_val_loss:.4f}' if best_val_loss < float('inf') else 'N/A',
            'lr': f'{current_lr:.2e}'
        })

    elapsed = time.time() - t0
    print(f"\n[+] Huấn luyện hoàn tất xuất sắc sau {elapsed/60:.2f} phút!")
    print(f"[+] Best Validation Loss đạt được: {best_val_loss:.4f}")

    # Đảm bảo lưu mô hình tốt nhất nếu chưa có file
    if not os.path.exists(BEST_MODEL_PATH):
        if hasattr(trainer.model, "module"):
            clean_state = trainer.model.module.state_dict()
        else:
            clean_state = trainer.model.state_dict()
        torch.save(clean_state, BEST_MODEL_PATH)
    print(f"[+] Trọng số mô hình tối ưu đã lưu tại: {BEST_MODEL_PATH} ({os.path.getsize(BEST_MODEL_PATH)/1e6:.1f} MB)")

    # Lưu lịch sử huấn luyện dưới dạng file JSON
    history_file = os.path.join(os.path.dirname(BEST_MODEL_PATH), "training_history.json")
    with open(history_file, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(f"[+] Đã lưu lịch sử huấn luyện vào: {history_file}")

    # ==============================================================================
    # VẼ BIỂU ĐỒ THEO DÕI TRAIN/VAL LOSS & LEARNING RATE (LEARNING CURVES)
    # ==============================================================================
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Panel 1: Train Loss vs Val Loss
        if history['step']:
            ax1.plot(history['step'], history['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2.2)
        if history['val_step']:
            ax1.plot(history['val_step'], history['val_loss'], label='Validation Loss', color='#d62728', marker='o', linewidth=2.0)
            if history['val_loss']:
                min_idx = np.argmin(history['val_loss'])
                best_s = history['val_step'][min_idx]
                best_l = history['val_loss'][min_idx]
                ax1.scatter([best_s], [best_l], color='gold', s=130, zorder=5, edgecolors='black', label=f'Best Val ({best_l:.4f})')
        
        ax1.set_xlabel('Iteration (Step)', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
        ax1.set_title('VietOCR Fine-Tuning Loss Curve', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper right', frameon=True)
        ax1.grid(True, linestyle='--', alpha=0.6)

        # Panel 2: Learning Rate Decay
        if history['step']:
            ax2.plot(history['step'], history['lr'], label='Learning Rate', color='#2ca02c', linewidth=2.2)
        ax2.set_xlabel('Iteration (Step)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
        ax2.set_title('Cosine Annealing Learning Rate Schedule', fontsize=14, fontweight='bold')
        ax2.legend(loc='upper right', frameon=True)
        ax2.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()
        curve_path = os.path.join(os.path.dirname(BEST_MODEL_PATH), "training_loss_curve.png")
        plt.savefig(curve_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"[+] Đã vẽ và lưu biểu đồ Loss Curve tại: {curve_path}")
    except Exception as pe:
        print(f"[!] Không thể render biểu đồ matplotlib: {pe}")

except Exception as e:
    print(f"[!] Có lỗi trong quá trình train: {e}")


## 8. Đánh Giá Hiệu Năng So Sánh Trước vs Sau Khi Fine-Tuning (Before & After)

In [ ]:
from vietocr.tool.predictor import Predictor

def compute_cer(gt: str, pred: str) -> float:
    import editdistance
    return editdistance.eval(gt, pred) / max(len(gt), 1)

# 1. Mô hình gốc Pretrained
base_cfg = Cfg.load_config_from_name('vgg_transformer')
base_cfg['device'] = DEVICE
base_predictor = Predictor(base_cfg)

# 2. Mô hình Fine-Tuned mới
if os.path.exists(BEST_MODEL_PATH):
    ft_cfg = Cfg.load_config_from_name('vgg_transformer')
    ft_cfg['device'] = DEVICE
    ft_cfg['weights'] = BEST_MODEL_PATH
    ft_predictor = Predictor(ft_cfg)
else:
    ft_predictor = base_predictor

print("[*] Đang đánh giá trên tập Validation...")
base_cers, ft_cers = [], []

for path, gt_label in val_samples[:30]:
    if not os.path.exists(path):
        continue
    crop_img = Image.open(path)
    
    pred_base = base_predictor.predict(crop_img)
    pred_ft = ft_predictor.predict(crop_img)
    
    cer_b = compute_cer(gt_label, pred_base)
    cer_f = compute_cer(gt_label, pred_ft)
    base_cers.append(cer_b)
    ft_cers.append(cer_f)

print("="*60)
print("BẢNG SO SÁNH HIỆU QUẢ FINE-TUNING (VALIDATION SET)")
print("="*60)
print(f"Pretrained Base VietOCR Mean CER: {np.mean(base_cers):.4f}")
print(f"Fine-Tuned VietOCR Mean CER:      {np.mean(ft_cers):.4f}")
improvement = (np.mean(base_cers) - np.mean(ft_cers)) / max(np.mean(base_cers), 1e-6) * 100
print(f"Mức độ cải thiện độ chính xác:    +{improvement:.1f}%")
print("="*60)

## 9. Hướng Dẫn Tích Hợp Trọng Số Mới Vào `testOCR.ipynb`

Sau khi chạy notebook này và có file `vietocr_chat_best.pth`:
Trong notebook `testOCR.ipynb`, tại **Cell 9 (`init_vietocr`)**, bạn chỉ cần cập nhật nạp trọng số mới:

```python
config = Cfg.load_config_from_name('vgg_transformer')
config['weights'] = '/kaggle/working/vietocr_chat_best.pth'  # Nạp trọng số đã fine-tune
config['device'] = DEVICE
vietocr_predictor = Predictor(config)
```

Khi đó, toàn bộ các lỗi về dấu thanh, font chữ mobile, và thuật ngữ công nghệ/tài chính sẽ được xử lý triệt để!